In [ ]:
import json
from typing import TypedDict, Annotated, Sequence, Optional
import operator
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv
from tavily import TavilyClient

# ====================== 配置区域 ======================
load_dotenv()

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
LLM_MODEL = "qwen3.8-max"
MAX_REFLECT_ROUND = 3  # 最多重试搜索3轮，防止死循环
tavily = TavilyClient(api_key=TAVILY_API_KEY)

# ====================== 数据结构定义 ======================
class TaskItem(BaseModel):
    task_name: str
    tool_name: str
    arguments: dict

class PlanOutput(BaseModel):
    tasks: list[TaskItem] = Field(description="有序任务列表")

class ReflectionOutput(BaseModel):
    is_sufficient: bool
    is_data_valid: bool
    problem: str
    next_action: Literal["FINISH","ADD_TASK","REPLAN"]
    new_tasks: list[TaskItem]

# Agent全局状态
class AgentState(TypedDict):
    user_query: str
    today_date: Optional[str]
    db_hold_data: Optional[dict]
    search_records: list[dict]
    task_queue: Sequence[TaskItem]
    completed_tasks: Sequence[TaskItem]
    reflect_records: Sequence[ReflectionOutput]
    reflect_round: int
    final_answer: Optional[str]

# ====================== 模拟工具实现（自行替换真实逻辑） ======================
def get_today_date() -> str:
    """获取今日日期"""
    return "2026-08-18"

def query_gold_db(name: str) -> dict:
    """查询黄金持仓数据库"""
    # 示例：{"name":"小红","quantity":1,"unit":"kg"}
    return {"name": name, "quantity": 1, "unit": "kg"}

def tavily_search(query: str) -> dict:
    """金价搜索"""
    resp = tavily.search(query=query, max_results=2)
    return resp

# 工具路由映射
tool_mapping = {
    "get_today_date": get_today_date,
    "query_gold_db": query_gold_db,
    "tavily_search": tavily_search
}

# ====================== Prompt模板 ======================
PLANNER_PROMPT = ChatPromptTemplate.from_messages([
    SystemMessage(content="""
你是任务规划器，根据用户黄金估值需求生成有序工具任务。
可用工具清单：
1. get_today_date：无参数，获取今日日期 YYYY-MM-DD
2. query_gold_db：参数 name[str]，查询人物黄金持仓
3. tavily_search：参数 query[str]，搜索当日人民币金价

约束：
1. 任务必须满足依赖顺序；
2. 禁止生成无法执行的任务；
输出严格遵循PlanOutput格式。
"""),
    HumanMessage(content="{user_query}")
])

REFLECTOR_PROMPT = ChatPromptTemplate.from_messages([
    SystemMessage(content="""
你是结果反思校验专家，严格按照规则审核全部执行信息：
校验清单：
1. 是否拿到正确当日日期；
2. 是否成功查询人物黄金持仓；kg统一换算为克；
3. 金价等级规则：
【等级1 首选】上金所AU9999、上金所金条、中钞国鼎、银行投资金条价格
【等级2 备选】黄金回收价格
【等级3 禁止用于金条估值】品牌金店首饰零售价（>1100元/克饰品价）
【等级4 禁止】美元国际金价、港币金价
4. 报价日期必须和今日日期匹配，过期价格不可采信

决策分支：
FINISH：信息齐全、数据合规，直接生成答案
ADD_TASK：新增搜索任务（更换query重新搜索金价）
REPLAN：全部任务作废，重新规划

输出严格按照ReflectionOutput JSON结构。
历史上下文：
{context}
"""),
])

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    SystemMessage(content="""
按照固定格式输出黄金估值结果：
人物：{name}
持有黄金：{total_g} 克（原始单位自动换算）
选用金价：{price} 元/克（价格类型：{price_type}）
预估资产价值：{value:.2f} 元

补充说明：本次估值采用投资金条基准价格；品牌黄金首饰零售价溢价更高，不适用于实物金条资产测算。
""")
])

# ====================== Graph节点定义 ======================
def planner_node(state: AgentState, llm):
    parser = PydanticOutputParser(pydantic_object=PlanOutput)
    chain = PLANNER_PROMPT | llm | parser
    result: PlanOutput = chain.invoke({"user_query": state["user_query"]})
    return {
        "task_queue": result.tasks,
        "completed_tasks": [],
        "reflect_records": [],
        "reflect_round": 0,
        "search_records": [],
        "final_answer": None
    }

def executor_node(state: AgentState, llm):
    task_queue = list(state["task_queue"])
    if not task_queue:
        return state

    # 取出队首任务执行
    current_task = task_queue.pop(0)
    func = tool_mapping[current_task.tool_name]
    output = func(**current_task.arguments)

    # 根据工具类型回填状态
    if current_task.tool_name == "get_today_date":
        state["today_date"] = output
    elif current_task.tool_name == "query_gold_db":
        state["db_hold_data"] = output
    elif current_task.tool_name == "tavily_search":
        state["search_records"].append({"query": current_task.arguments["query"], "result": output})

    # 更新队列 & 已完成任务
    new_completed = list(state["completed_tasks"]) + [current_task]
    return {"task_queue": task_queue, "completed_tasks": new_completed}

def reflector_node(state: AgentState, llm):
    if state["reflect_round"] >= MAX_REFLECT_ROUND:
        # 达到最大轮次，强制结束
        return ReflectionOutput(
            is_sufficient=False,
            is_data_valid=False,
            problem="达到最大搜索重试次数，无法获取有效金价",
            next_action="FINISH",
            new_tasks=[]
        )

    # 组装全部上下文送入反思器
    context = json.dumps({
        "user_query": state["user_query"],
        "today_date": state["today_date"],
        "db_hold_data": state["db_hold_data"],
        "search_records": state["search_records"],
        "completed_tasks": [t.model_dump() for t in state["completed_tasks"]]
    }, ensure_ascii=False, indent=2)

    parser = PydanticOutputParser(pydantic_object=ReflectionOutput)
    chain = REFLECTOR_PROMPT | llm | parser
    reflect_result: ReflectionOutput = chain.invoke({"context": context})

    new_reflect_round = state["reflect_round"] + 1
    new_reflect_history = list(state["reflect_records"]) + [reflect_result]
    return {
        "reflect_records": new_reflect_history,
        "reflect_round": new_reflect_round,
        "task_queue": reflect_result.new_tasks if reflect_result.next_action in ("ADD_TASK", "REPLAN") else state["task_queue"]
    }

def generate_answer_node(state: AgentState, llm):
    db_data = state["db_hold_data"]
    search_info = state["search_records"]
    today = state["today_date"]

    # LLM汇总生成最终文本
    msg = HumanMessage(content=f"""
    用户问题：{state['user_query']}
    今日日期：{today}
    持仓数据：{json.dumps(db_data, ensure_ascii=False)}
    所有搜索结果：{json.dumps(search_info, ensure_ascii=False)}
    结合金价筛选规则，计算并输出标准化估值结果；若无有效数据如实说明，禁止编造价格。
    """)
    resp = llm.invoke([ANSWER_PROMPt, msg])
    return {"final_answer": resp.content}

# ====================== 构建图与路由 ======================
def build_graph(llm):
    graph = StateGraph(AgentState)

    graph.add_node("planner", lambda s: planner_node(s, llm))
    graph.add_node("executor", lambda s: executor_node(s, llm))
    graph.add_node("reflector", lambda s: reflector_node(s, llm))
    graph.add_node("generate_answer", lambda s: generate_answer_node(s, llm))

    graph.set_entry_point("planner")
    graph.add_edge("planner", "executor")
    graph.add_edge("executor", "reflector")

    # 反思器路由判断
    def route_reflector(state: AgentState):
        last_reflect = state["reflect_records"][-1] if state["reflect_records"] else None
        if not last_reflect or last_reflect.next_action == "FINISH":
            return "generate_answer"
        else:
            return "executor"

    graph.add_conditional_edges("reflector", route_reflector, {
        "executor": "executor",
        "generate_answer": "generate_answer"
    })
    graph.add_edge("generate_answer", END)
    return graph.compile()

# ====================== 入口调用示例 ======================
if __name__ == "__main__":
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model=LLM_MODEL,
        api_key="你的模型key",
        base_url="模型接口地址"
    )
    agent_graph = build_graph(llm)

    inputs = {
        "user_query": "小红现在持有的黄金市值多少钱？"
    }
    result = agent_graph.invoke(inputs)
    print(result["final_answer"])
